# 面试问题：Adaptive Test-Time Compute 怎样按问题难度和风险分配推理预算？

可以直接复述的回答是：第一，不应给所有请求固定相同步数。第二，先用难度、风险和截止时间分配最大预算。第三，每一步产生候选、置信度和可验证信号。第四，低风险且验证通过可提前停止，高风险还需要最小审议步数。第五，预算耗尽仍未验证时应弃答或转人工。第六，用同一批请求比较准确率、平均步数、成本和弃答率。下面用五个客服与运维任务的确定性推理轨迹演示。

## 真实案例：客服与运维 Agent 的动态思考预算

五条脱敏任务覆盖密码重置、退款规则、生产故障、SQL 汇总和缺少司法辖区的合同问题。每条预先保存逐步候选、置信度和 verifier 结果，模拟真实模型在增加计算后的状态。轨迹是教学构造，用于验证调度策略，不代表真实模型的 chain-of-thought。

In [1]:
cases = [  # 定义五条具有不同难度、风险和验证轨迹的任务
    {"id": "TC-01", "question": "怎样重置已验证员工的 VPN 密码？", "difficulty": 1, "risk": "low", "deadline_ms": 500, "expected": "发送自助重置链接", "steps": [("发送自助重置链接", 0.91, True), ("发送自助重置链接", 0.96, True)]},  # 简单且第一步可验证
    {"id": "TC-02", "question": "已签收商品 8 天后申请退款，规则如何判断？", "difficulty": 3, "risk": "medium", "deadline_ms": 1200, "expected": "检查品类例外后转人工", "steps": [("直接退款", 0.72, False), ("超过七天拒绝", 0.78, False), ("检查品类例外后转人工", 0.88, True)]},  # 需要三步组合规则
    {"id": "TC-03", "question": "生产支付错误率突然升到 60%，下一步？", "difficulty": 4, "risk": "high", "deadline_ms": 1600, "expected": "冻结发布并启动事故响应", "steps": [("继续观察", 0.94, False), ("扩容实例", 0.81, False), ("检查发布变更", 0.84, False), ("冻结发布并启动事故响应", 0.93, True)]},  # 高置信错误需要风险最小步数和验证
    {"id": "TC-04", "question": "SQL 怎样汇总每个租户的订单金额？", "difficulty": 2, "risk": "low", "deadline_ms": 800, "expected": "GROUP BY tenant_id", "steps": [("ORDER BY tenant_id", 0.66, False), ("GROUP BY tenant_id", 0.92, True), ("GROUP BY tenant_id", 0.95, True)]},  # 两步可验证代码知识
    {"id": "TC-05", "question": "这份跨境合同一定合法吗？", "difficulty": 5, "risk": "high", "deadline_ms": 1800, "expected": "abstain", "steps": [("合法", 0.61, False), ("需要司法辖区", 0.70, False), ("可能合法", 0.73, False), ("信息不足", 0.79, False), ("转法务", 0.82, False)]},  # 缺少关键事实应耗尽预算后弃答
]  # 结束五条动态预算输入
print("任务输入：id | difficulty | risk | deadline | question")  # 展示预算器读取的真实字段
for case in cases:  # 逐条输出五个任务
    print(f"{case['id']} | {case['difficulty']} | {case['risk']:6} | {case['deadline_ms']:4}ms | {case['question']}")  # 呈现难度、风险和时限差异


任务输入：id | difficulty | risk | deadline | question
TC-01 | 1 | low    |  500ms | 怎样重置已验证员工的 VPN 密码？
TC-02 | 3 | medium | 1200ms | 已签收商品 8 天后申请退款，规则如何判断？
TC-03 | 4 | high   | 1600ms | 生产支付错误率突然升到 60%，下一步？
TC-04 | 2 | low    |  800ms | SQL 怎样汇总每个租户的订单金额？
TC-05 | 5 | high   | 1800ms | 这份跨境合同一定合法吗？


## Baseline / 基线：所有请求固定两步并按最高置信停止

固定预算简单可预测，但简单请求浪费一步，复杂请求又不够。Baseline 不检查 verifier，只返回两步内置信度最高的候选。

In [2]:
fixed_budget = 2  # 为所有请求设置相同两步预算
baseline_rows = []  # 收集固定预算的答案、步数和正确性
print("固定预算：id | selected_answer | confidence | steps | correct")  # 输出逐任务基线结果
for case in cases:  # 对五个任务统一运行两步
    available = case["steps"][:fixed_budget]  # 截取固定数量的推理候选
    selected = max(available, key=lambda step: step[1])  # 只按自报置信度选择候选
    correct = selected[0] == case["expected"]  # 与离线人工期望比较
    baseline_rows.append({"id": case["id"], "answer": selected[0], "confidence": selected[1], "steps": len(available), "correct": correct})  # 保存基线结果
    print(f"{case['id']} | {selected[0]} | {selected[1]:.2f} | {len(available)} | {correct}")  # 展示高置信错误和预算不足
baseline_accuracy = sum(row["correct"] for row in baseline_rows) / len(baseline_rows)  # 计算固定预算准确率
baseline_average_steps = sum(row["steps"] for row in baseline_rows) / len(baseline_rows)  # 计算固定两步平均成本
print(f"固定预算：accuracy={baseline_accuracy:.1%}，average_steps={baseline_average_steps:.2f}")  # 输出质量与成本基线


固定预算：id | selected_answer | confidence | steps | correct
TC-01 | 发送自助重置链接 | 0.96 | 2 | True
TC-02 | 超过七天拒绝 | 0.78 | 2 | False
TC-03 | 继续观察 | 0.94 | 2 | False
TC-04 | GROUP BY tenant_id | 0.92 | 2 | True
TC-05 | 需要司法辖区 | 0.70 | 2 | False
固定预算：accuracy=40.0%，average_steps=2.00


## 核心实现：风险感知预算、Verifier 与提前停止

难度决定最大步数，高风险至少审议三步。只有 verifier 通过且达到置信阈值才可停止；预算耗尽未通过则返回 abstain。

In [3]:
risk_min_steps = {"low": 1, "medium": 2, "high": 3}  # 定义不同风险的最小审议步数
confidence_thresholds = {"low": 0.85, "medium": 0.85, "high": 0.90}  # 高风险使用更严格置信门槛
def allocate_budget(case):  # 根据难度、风险和截止时间分配最大推理步数
    difficulty_budget = max(1, min(case["difficulty"], len(case["steps"])))  # 把难度映射到可用轨迹范围
    deadline_budget = max(1, min(len(case["steps"]), case["deadline_ms"] // 350))  # 用每步 350ms 代理约束时限
    minimum = risk_min_steps[case["risk"]]  # 获取当前风险必须覆盖的最小步数
    return min(len(case["steps"]), max(minimum, min(difficulty_budget, deadline_budget)))  # 在时限代理和风险下得到最终预算
def adaptive_solve(case):  # 执行逐步验证和风险感知提前停止
    budget = allocate_budget(case)  # 获取当前请求的最大计算预算
    trace = []  # 保存每一步候选、置信度和验证信号
    for index, (answer, confidence, verified) in enumerate(case["steps"][:budget], start=1):  # 按预算顺序读取推理候选
        trace.append({"step": index, "answer": answer, "confidence": confidence, "verified": verified})  # 写入可审计状态而非隐藏思维文本
        enough_deliberation = index >= risk_min_steps[case["risk"]]  # 检查风险要求的最小步数
        confident = confidence >= confidence_thresholds[case["risk"]]  # 检查当前候选置信门槛
        if verified and enough_deliberation and confident:  # 三项条件均满足时才提前停止
            return {"answer": answer, "steps": index, "budget": budget, "status": "answered", "trace": trace}  # 返回有验证证据的答案
    return {"answer": "abstain", "steps": len(trace), "budget": budget, "status": "abstained", "trace": trace}  # 预算耗尽后安全弃答
focus_result = adaptive_solve(cases[2])  # 对高风险支付事故执行动态推理
print("TC-03 自适应轨迹：step | answer | confidence | verified")  # 输出高置信错误到正确动作的中间过程
for step in focus_result["trace"]:  # 逐步展示四个候选状态
    print(f"{step['step']} | {step['answer']} | {step['confidence']:.2f} | {step['verified']}")  # 让预算增加为何必要可见
print("TC-03 最终：", {key: focus_result[key] for key in ("answer", "steps", "budget", "status")})  # 展示风险门禁后的决定


TC-03 自适应轨迹：step | answer | confidence | verified
1 | 继续观察 | 0.94 | False
2 | 扩容实例 | 0.81 | False
3 | 检查发布变更 | 0.84 | False
4 | 冻结发布并启动事故响应 | 0.93 | True
TC-03 最终： {'answer': '冻结发布并启动事故响应', 'steps': 4, 'budget': 4, 'status': 'answered'}


## 失败案例与修正：高置信不等于正确

TC-03 第一步“继续观察”置信度 0.94，高于最终正确动作前的多数候选。只按 confidence 会立即停止；Verifier 与高风险最小步数阻止这次危险早停。

In [4]:
incident = cases[2]  # 取出生产支付故障作为高风险反例
naive_first = incident["steps"][0]  # 读取第一步高置信错误候选
naive_stop = naive_first[1] >= 0.90  # 模拟只看置信度的早停策略
naive_answer = naive_first[0] if naive_stop else "continue"  # 生成天真策略动作
safe_answer = focus_result["answer"]  # 读取 verifier 和风险预算后的动作
saved_risk = naive_answer != incident["expected"] and safe_answer == incident["expected"]  # 判断修正是否避免危险动作
print(f"修正前：answer={naive_answer}，confidence={naive_first[1]:.2f}，verified={naive_first[2]}")  # 展示高置信错误
print(f"修正后：answer={safe_answer}，steps={focus_result['steps']}，status={focus_result['status']}")  # 展示额外计算和验证后的正确动作
print("是否避免高风险错误：", saved_risk)  # 明确输出门禁价值


修正前：answer=继续观察，confidence=0.94，verified=False
修正后：answer=冻结发布并启动事故响应，steps=4，status=answered
是否避免高风险错误： True


## 结果表：五条任务的准确率、成本与弃答

In [5]:
adaptive_rows = []  # 收集五条请求的动态预算结果
print("id | expected | baseline_answer/steps | adaptive_answer/steps/budget | status | correct")  # 输出逐任务同数据对照
for case, baseline in zip(cases, baseline_rows):  # 对齐固定和动态两种策略
    result = adaptive_solve(case)  # 执行当前任务的风险感知预算
    correct = result["answer"] == case["expected"]  # 把安全弃答也按人工期望计为正确决定
    adaptive_rows.append({"id": case["id"], "answer": result["answer"], "steps": result["steps"], "budget": result["budget"], "status": result["status"], "correct": correct})  # 保存完整结果
    print(f"{case['id']} | {case['expected']} | {baseline['answer']}/{baseline['steps']} | {result['answer']}/{result['steps']}/{result['budget']} | {result['status']} | {correct}")  # 展示计算分配差异
adaptive_accuracy = sum(row["correct"] for row in adaptive_rows) / len(adaptive_rows)  # 计算动态策略决策准确率
adaptive_average_steps = sum(row["steps"] for row in adaptive_rows) / len(adaptive_rows)  # 计算动态策略平均计算步数
abstention_rate = sum(row["status"] == "abstained" for row in adaptive_rows) / len(adaptive_rows)  # 计算预算耗尽弃答率
print(f"汇总：baseline_accuracy={baseline_accuracy:.1%}，adaptive_accuracy={adaptive_accuracy:.1%}，adaptive_steps={adaptive_average_steps:.2f}，abstention={abstention_rate:.1%}")  # 输出质量、成本和覆盖权衡


id | expected | baseline_answer/steps | adaptive_answer/steps/budget | status | correct
TC-01 | 发送自助重置链接 | 发送自助重置链接/2 | 发送自助重置链接/1/1 | answered | True
TC-02 | 检查品类例外后转人工 | 超过七天拒绝/2 | 检查品类例外后转人工/3/3 | answered | True
TC-03 | 冻结发布并启动事故响应 | 继续观察/2 | 冻结发布并启动事故响应/4/4 | answered | True
TC-04 | GROUP BY tenant_id | GROUP BY tenant_id/2 | GROUP BY tenant_id/2/2 | answered | True
TC-05 | abstain | 需要司法辖区/2 | abstain/5/5 | abstained | True
汇总：baseline_accuracy=40.0%，adaptive_accuracy=100.0%，adaptive_steps=3.00，abstention=20.0%


## 结果解读

TC-01 在第一步验证通过，动态策略避免固定预算的多余一步；TC-02、TC-03、TC-04 分别得到与难度相符的三、四、两步预算。TC-05 即使走到最大预算仍没有验证证据，因此返回 abstain。关键不是“永远想更久”，而是把额外计算投给高不确定或高风险请求。

## 生产边界

生产实现需要在线难度预测、真实 token/延迟成本、模型校准、外部 verifier、并发队列和总租户预算。隐藏 chain-of-thought 不应进入日志，只记录候选摘要、验证结果和停止原因。阈值需在独立数据上校准，并防止高风险请求耗尽系统容量。本例轨迹是预设的确定性教学数据。

## 最小回归测试

In [6]:
assert len(cases) >= 5  # 保证动态计算案例覆盖五种难度和风险任务
assert naive_stop is True and saved_risk is True  # 保证高置信错误被 verifier 与风险预算修正
assert focus_result["answer"] == incident["expected"] and focus_result["steps"] == 4  # 保证生产事故使用足够计算后得到正确动作
assert next(row for row in adaptive_rows if row["id"] == "TC-01")["steps"] == 1  # 保证简单请求可以提前停止节省计算
assert next(row for row in adaptive_rows if row["id"] == "TC-05")["status"] == "abstained"  # 保证高风险无证据问题耗尽预算后弃答
assert adaptive_accuracy > baseline_accuracy  # 保证动态分配在同一五题上提高决策准确率
